# Day 18 — Filtering inside aggregations

*100 Days of Polars · Phase 2: Expressions*

- **Post:** {{TODO: Medium link}}
- **Dataset:** [Olist Brazilian e-commerce](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) — CSVs in `data/raw/`
- **Written for:** Polars 2.x (run from the repo root)

In [1]:
import polars as pl

print("polars", pl.__version__)

polars 2.0.0


`filter` is not only a context. As an **expression**, `pl.col(x).filter(cond)` keeps the values where `cond` is true, so you can compute several conditional aggregates in **one pass** over each group.
SQL calls this `SUM(CASE WHEN ...)` or `SUM(x) FILTER (WHERE ...)`.

In [2]:
items = pl.read_csv("data/raw/olist_order_items_dataset.csv", try_parse_dates=True)
payments = pl.read_csv("data/raw/olist_order_payments_dataset.csv")

## 1. Conditional aggregates over the whole table

In [3]:
items.select(
    revenue=pl.col("price").sum().round(2),
    revenue_over_500=pl.col("price").filter(pl.col("price") > 500).sum().round(2),
    n_over_500=pl.col("price").filter(pl.col("price") > 500).len(),
)

revenue,revenue_over_500,n_over_500
f64,f64,u32
1.3592e7,2.9822e6,3216


## 2. Per group: payment mix per order

In [4]:
pay_mix = payments.group_by("order_id").agg(
    total=pl.col("payment_value").sum(),
    card=pl.col("payment_value").filter(pl.col("payment_type") == "credit_card").sum(),
    boleto=pl.col("payment_value").filter(pl.col("payment_type") == "boleto").sum(),
    voucher=pl.col("payment_value").filter(pl.col("payment_type") == "voucher").sum(),
    n_methods=pl.col("payment_type").n_unique(),
)
pay_mix.sort("n_methods", descending=True).head(5)

order_id,total,card,boleto,voucher,n_methods
str,f64,f64,f64,f64,u32
"""0c3e2dc4a3eeb282bd4df0b620dcff…",83.68,59.72,0.0,23.96,2
"""0fd20e900b40f5cb8e6fd31df27aab…",315.4,65.4,0.0,250.0,2
"""29dba01f92da927e611d340bc135f6…",884.2,734.2,0.0,150.0,2
"""3df55fc07ff463109ce0422439693a…",316.6,298.51,0.0,18.09,2
"""c54ebaa4a93a3de78a932ba2ce323f…",105.37,55.37,0.0,50.0,2


In [5]:
# How many orders used more than one payment method?
print(pay_mix.filter(pl.col("n_methods") > 1).height, "orders used more than one payment method")

2246 orders used more than one payment method


An empty selection sums to `0`, so every order gets a value for every method.

## 3. Row-wise totals: `sum_horizontal`

Now check the parts add up to the total, across columns.

In [6]:
pay_mix.with_columns(
    parts=pl.sum_horizontal("card", "boleto", "voucher"),
).with_columns(
    other=(pl.col("total") - pl.col("parts")).round(2),
).filter(pl.col("other") != 0).head(5)  # debit_card and not_defined are in 'other'

order_id,total,card,boleto,voucher,n_methods,parts,other
str,f64,f64,f64,f64,u32,f64,f64
"""4436546aaa7d7c56c61aa5ee6785ee…",22.29,0.0,0.0,0.0,1,0.0,22.29
"""f9eb516475260ab01f33937aac3cfd…",156.39,0.0,0.0,0.0,1,0.0,156.39
"""e3194d56283622975638628a739c9e…",67.83,0.0,0.0,0.0,1,0.0,67.83
"""d3454c86a9d5b5a519364756eea7e2…",149.14,0.0,0.0,0.0,1,0.0,149.14
"""33a3edb84b9df4cb49546859b990ac…",67.5,0.0,0.0,0.0,1,0.0,67.5


In [7]:
# sum_horizontal skips nulls by default; set ignore_nulls=False to propagate them
df = pl.DataFrame({"a": [1, None], "b": [2, 3]})
df.with_columns(
    skip=pl.sum_horizontal("a", "b"),
    strict=pl.sum_horizontal("a", "b", ignore_nulls=False),
)

a,b,skip,strict
i64,i64,i64,i64
1,2,3,3
null,3,3,null


## 4. Per seller: several conditions, one `group_by`

In [8]:
(
    items.group_by("seller_id")
    .agg(
        n_items=pl.len(),
        revenue=pl.col("price").sum(),
        high_freight_items=pl.col("price").filter(pl.col("freight_value") > pl.col("price")).len(),
        avg_price_cheap=pl.col("price").filter(pl.col("price") < 50).mean(),
    )
    .filter(pl.col("n_items") >= 100)
    .sort("high_freight_items", descending=True)
    .head(5)
)

seller_id,n_items,revenue,high_freight_items,avg_price_cheap
str,u32,f64,u32,f64
"""8b321bb669392f5163d04c59e235e0…",1018,17535.69,413,16.510881
"""92eb0f42c21942b6552362b9b11470…",365,6548.63,144,16.949141
"""128639473a139ac0f3e5f5ade55873…",560,11908.85,135,20.030888
"""b76dba6c951ab00dc4edf0a1aa8803…",169,2574.78,84,15.235385
"""d2374cbcbb3ca4ab1086534108cc3a…",631,21223.92,75,26.735925


## 5. Gotcha: `filter` vs `when` inside `mean`

`filter` **removes** values, so the mean is over matching rows only.
`when/then/otherwise(0)` **keeps** every row with a 0, so the mean is over all rows. Both are valid; they answer different questions.

In [9]:
items.select(
    avg_price_of_expensive=pl.col("price").filter(pl.col("price") > 500).mean(),
    avg_contribution_of_expensive=pl.when(pl.col("price") > 500).then(pl.col("price")).otherwise(0).mean(),
)

avg_price_of_expensive,avg_contribution_of_expensive
f64,f64
927.313924,26.473516


In [10]:
# And on an empty selection: sum -> 0, mean -> null
items.select(
    s=pl.col("price").filter(pl.col("price") < 0).sum(),
    m=pl.col("price").filter(pl.col("price") < 0).mean(),
)

s,m
f64,f64
0.0,null


## pandas comparison: code

No filter inside a named aggregation, so mask the values first. pandas' own `groupby().filter()` is something else: it keeps or drops whole groups.

In [11]:
# pandas, for comparison: mask first, then aggregate (one temporary column per condition)
pay_pd = payments.to_pandas()
pd_card = (
    pay_pd.assign(card=pay_pd["payment_value"].where(pay_pd["payment_type"] == "credit_card"))
    .groupby("order_id")
    .agg(total=("payment_value", "sum"), card=("card", "sum"))
)
print("same card total as Polars:", round(pd_card["card"].sum(), 2) == round(pay_mix["card"].sum(), 2))

same card total as Polars: True


### How pandas `groupby().filter()` works, step by step

It runs your function **once per group**, gets back a single `True`/`False`, and keeps **all the rows** of the groups that returned `True`. The result has the original rows and columns, not one row per group.

In [12]:
import pandas as pd

toy = pd.DataFrame({
    "payment_type": ["card", "card", "card", "boleto", "boleto", "voucher"],
    "value":        [10.0,   20.0,   30.0,   5.0,      15.0,     50.0],
})
toy

,payment_type,value
0,card,10.0
1,card,20.0
2,card,30.0
3,boleto,5.0
4,boleto,15.0
5,voucher,50.0


In [13]:
# Step 1: what the function sees. One sub-frame per group, and the answer it would give
for name, g in toy.groupby("payment_type"):
    print(f"{name:<8} rows={len(g)}  keep={len(g) >= 2}")

boleto   rows=2  keep=True
card     rows=3  keep=True
voucher  rows=1  keep=False


In [14]:
# Step 2: filter keeps ALL rows of the groups that answered True. Same columns, original rows
toy.groupby("payment_type").filter(lambda g: len(g) >= 2)

,payment_type,value
0,card,10.0
1,card,20.0
2,card,30.0
3,boleto,5.0
4,boleto,15.0


In [15]:
# The condition can be any per-group aggregate: here "groups whose total value is over 40"
toy.groupby("payment_type").filter(lambda g: g["value"].sum() > 40)

,payment_type,value
0,card,10.0
1,card,20.0
2,card,30.0
5,voucher,50.0


In [16]:
# Contrast: agg returns ONE ROW PER GROUP, not the original rows
toy.groupby("payment_type").agg(n=("value", "size"), total=("value", "sum"))

,n,total
payment_type,,
boleto,2,20.0
card,3,60.0
voucher,1,50.0


In [17]:
# Polars equivalent of groupby().filter: a filter with a window expression (covered later in the series)
pl.from_pandas(toy).filter(pl.len().over("payment_type") >= 2)

payment_type,value
str,f64
"""card""",10.0
"""card""",20.0
"""card""",30.0
"""boleto""",5.0
"""boleto""",15.0


In [18]:
# pandas groupby().filter keeps or drops WHOLE groups by a per-group condition (returns rows, not an aggregate)
kept_pd = pay_pd.groupby("payment_type").filter(lambda g: len(g) >= 100)
kept_pl = payments.filter(pl.len().over("payment_type") >= 100)   # Polars: a window expression (later in the series)
print(len(pay_pd), "->", len(kept_pd), "rows | dropped types:", sorted(set(pay_pd["payment_type"]) - set(kept_pd["payment_type"])))
print("same rows in Polars:", kept_pl.height == len(kept_pd))

103886 -> 103883 rows | dropped types: ['not_defined']
same rows in Polars: True


## 6. pandas comparison

In pandas this is usually one `groupby` per condition plus a merge, or a slow `apply`. In Polars it is one `agg` and every expression runs in parallel.

## Try it yourself

1. Per seller: revenue from items shipped (by `shipping_limit_date`) in 2017 vs 2018, in one `agg`.
2. Per order: number of installments for card payments only.
3. Share of each order's value paid by voucher, then the average of that share across orders that used a voucher.